## NB-06 — Query Expansion and Re-Ranking

ShopSmart India shoppers use highly variable language: "earphones", "earbuds",
"TWS", "wireless audio", "kanphone" (Hinglish). A single query embedding will
miss products described with a different but equivalent term. This notebook
teaches query expansion — rewriting each query into multiple variants using
Claude — and cross-encoder re-ranking to improve the precision of comparative
recommendation results. Together these two techniques address the two most
common retrieval failures in the CatalogueIQ demo.

### Concepts Covered

- Why synonym diversity in ShopSmart India queries breaks single-embedding retrieval
- Building a domain-specific synonym expansion dictionary for Indian e-commerce
- Using Claude to expand each query into 3 semantically equivalent variants
- Multi-query retrieval: embedding each variant and merging results with deduplication
- Cross-encoder re-ranking with `cross-encoder/ms-marco-MiniLM-L-6-v2`
- Measuring the impact of expansion: distinct products in top-5 before vs. after expansion
- When expansion hurts: cases where variant queries introduce off-topic retrievals

In [ ]:
# ── SETUP ──────────────────────────────────────────────────────
import os
import json
import pickle
import re
from pathlib import Path
from dotenv import load_dotenv
import faiss
import numpy as np
from langchain.schema import Document
from langchain_anthropic import ChatAnthropic
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS as LangchainFAISS
from sentence_transformers import SentenceTransformer, CrossEncoder

load_dotenv()

MODEL = "claude-sonnet-4-6"
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
RERANKER_MODEL  = "cross-encoder/ms-marco-MiniLM-L-6-v2"
INDEX_DIR = Path("data/index")
os.makedirs("output", exist_ok=True)

assert os.getenv("ANTHROPIC_API_KEY"), "ANTHROPIC_API_KEY not found in .env"
print(f"LLM: {MODEL}")
print(f"Embedding model: {EMBEDDING_MODEL}")
print(f"Cross-encoder: {RERANKER_MODEL}")


### Step 1 — ShopSmart India Synonym Expansion Dictionary

Before using Claude to expand queries dynamically, we build a static synonym
dictionary for the highest-frequency query terms in the ShopSmart India domain.
This dictionary is used as a fallback and as a seed for the Claude expander.

In [ ]:
# ── DOMAIN SYNONYM DICTIONARY ────────────────────────────────

# These synonyms were validated in NB-01 using cosine similarity.
# earphones ↔ earbuds had high similarity; we add the full set here.

SHOPSMRT_SYNONYMS = {
    # Audio product synonyms (critical for product factual and comparative queries)
    "earphones": ["earbuds", "TWS earbuds", "in-ear headphones", "wireless earphones"],
    "earbuds":   ["earphones", "TWS", "in-ear headphones", "wireless audio"],
    "headphones":["over-ear headphones", "headset", "cans", "audio headset"],

    # Mobile/tech synonyms
    "mobile":    ["smartphone", "phone", "handset", "cell phone"],
    "mobile phone": ["smartphone", "handset", "Android phone", "cellphone"],

    # Fashion synonyms (important for Fashion category queries)
    "kurtis":    ["kurta", "ethnic wear", "Indian wear", "kurti tops"],
    "kurta":     ["kurti", "ethnic top", "Indian ethnic wear"],
    "saree":     ["sari", "Indian saree", "ethnic saree"],

    # Return/policy synonyms (important for policy queries)
    "returns":   ["refund", "exchange", "send back", "return policy"],
    "refund":    ["money back", "return", "reimbursement"],
    "exchange":  ["replacement", "swap", "return for exchange"],

    # Price/value synonyms (common in comparative queries)
    "cheap":     ["affordable", "budget", "low price", "value for money", "sasta"],
    "best":      ["top rated", "highly rated", "recommended", "popular"],

    # Noise cancellation (very common in audio queries)
    "noise cancellation": ["ANC", "active noise cancelling", "noise cancelling"],
    "ANC":       ["active noise cancellation", "noise cancelling", "noise isolation"],
}

def apply_synonym_expansion(query: str) -> list[str]:
    """
    Apply static synonym expansion to a query.
    Returns the original query plus one expanded variant per matched term.
    """
    variants = [query]
    query_lower = query.lower()

    for term, synonyms in SHOPSMRT_SYNONYMS.items():
        if term.lower() in query_lower:
            for synonym in synonyms[:2]:    # take first 2 synonyms per match
                expanded = re.sub(
                    re.escape(term), synonym, query, flags=re.IGNORECASE
                )
                if expanded not in variants:
                    variants.append(expanded)

    return variants[:4]     # cap at 4 variants to control API cost

# Test the static expander
test_q = "best earphones under 2000 rupees with noise cancellation"
print(f"Original: {test_q}")
print("Expanded variants:")
for v in apply_synonym_expansion(test_q):
    print(f"  → {v}")


### Step 2 — Claude-Powered Query Expansion

For queries that don't match the static dictionary, we use Claude to generate
3 semantically equivalent variants. This handles Hinglish terms, abbreviations,
and novel product categories automatically.

In [ ]:
# COST NOTE: Each expand_with_llm() call makes one API call to Claude.
# With 3 queries in this notebook: ~$0.02 total.

llm = ChatAnthropic(model=MODEL, temperature=0.3, max_tokens=512)

def expand_with_llm(query: str, n_variants: int = 3) -> list[str]:
    """
    Use Claude to generate n_variants semantically equivalent rewrites
    of the query in the ShopSmart India e-commerce context.
    Returns the original query plus LLM-generated variants.
    """
    prompt = f"""You are an Indian e-commerce search expert.
Rewrite the following shopper query into {n_variants} alternative phrasings
that preserve the exact same intent but use different vocabulary.
Consider: product synonyms, Indian English, Hinglish, technical abbreviations.

Query: {query}

Respond ONLY with a JSON array of {n_variants} strings. Example:
["variant 1", "variant 2", "variant 3"]
No explanation. No markdown. Just the JSON array."""

    # COST NOTE: API call to Claude
    response = llm.invoke(prompt)
    raw = response.content.strip()

    try:
        # Strip markdown fences if model adds them despite instructions
        raw = re.sub(r"```json|```", "", raw).strip()
        variants = json.loads(raw)
        if isinstance(variants, list):
            return [query] + [v for v in variants if v != query]
    except json.JSONDecodeError:
        pass

    # Fallback: return original + static expansion
    return apply_synonym_expansion(query)

# Test LLM expansion on three key ShopSmart India query types
test_queries = [
    "best earbuds under 2000 rupees ANC battery life",
    "can I return kurta bought on sale",
    "smartwatch stopped working warranty claim",
]

print("─" * 65)
for q in test_queries:
    # COST NOTE: API call to Claude
    variants = expand_with_llm(q, n_variants=3)
    print(f"\nOriginal: {q}")
    for i, v in enumerate(variants[1:], 1):
        print(f"  Variant {i}: {v}")


### Step 3 — Multi-Query Retrieval with Deduplication

For each query we retrieve `top_k` results per variant, then deduplicate by
`product_id` (for product chunks) or `page_content` hash (for policy chunks),
and keep the highest-scoring occurrence of each unique document.

In [ ]:
# ── LOAD INDEX AND BUILD VECTORSTORE ─────────────────────────

index_path = INDEX_DIR / "faiss.index"
meta_path  = INDEX_DIR / "metadata.pkl"

if not index_path.exists():
    raise FileNotFoundError("Run NB-04 first to build the FAISS index.")

faiss_index = faiss.read_index(str(index_path))
with open(meta_path, "rb") as f:
    all_docs = pickle.load(f)

hf_embedder = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)
vectorstore = LangchainFAISS.from_documents(all_docs, hf_embedder)

def multi_query_retrieve(
    query: str,
    use_llm_expansion: bool = True,
    top_k_per_variant: int = 3,
) -> list[tuple[Document, float]]:
    """
    Expand query → retrieve top_k per variant → deduplicate → return ranked list.
    """
    if use_llm_expansion:
        # COST NOTE: One API call to Claude per call to expand_with_llm
        variants = expand_with_llm(query, n_variants=2)
    else:
        variants = apply_synonym_expansion(query)[:3]

    seen_ids = {}  # key: unique doc identifier; value: (doc, score)

    for variant in variants:
        results = vectorstore.similarity_search_with_score(variant, k=top_k_per_variant)
        for doc, score in results:
            # Unique key: product_id for catalogue rows, content hash for others
            pid = doc.metadata.get("product_id", "")
            unique_key = pid if pid else hash(doc.page_content[:100])
            # Keep the highest-scoring (lowest L2 distance) occurrence
            if unique_key not in seen_ids or score < seen_ids[unique_key][1]:
                seen_ids[unique_key] = (doc, score)

    # Sort by score ascending (FAISS returns L2 distances, lower = more similar)
    ranked = sorted(seen_ids.values(), key=lambda x: x[1])
    return ranked

print("Multi-query retrieval function ready.")


### Step 4 — Cross-Encoder Re-Ranking

The bi-encoder (SentenceTransformer) retrieves candidates efficiently but may
rank them imprecisely. A cross-encoder reads the query and each candidate
together, producing a relevance score that is more accurate but slower.
We use it as a second-stage reranker on the merged candidate set.

In [ ]:
# ── CROSS-ENCODER RE-RANKER ──────────────────────────────────

print(f"Loading cross-encoder: {RERANKER_MODEL}")
cross_encoder = CrossEncoder(RERANKER_MODEL)

def rerank(query: str, candidates: list[tuple[Document, float]],
           top_n: int = 5) -> list[tuple[Document, float]]:
    """
    Re-rank candidate (Document, score) pairs using the cross-encoder.
    Returns top_n results sorted by cross-encoder score (higher = more relevant).
    """
    if not candidates:
        return []
    pairs = [(query, doc.page_content[:512]) for doc, _ in candidates]
    # Cross-encoder returns a relevance score for each (query, doc) pair
    ce_scores = cross_encoder.predict(pairs)
    # Combine with original bi-encoder rank, then sort by CE score descending
    reranked = sorted(
        zip([d for d, _ in candidates], ce_scores),
        key=lambda x: x[1], reverse=True
    )
    return reranked[:top_n]

print("Cross-encoder loaded and ready.")


### Step 5 — Before vs After Comparison

We compare the comparative recommendation query with and without expansion and
re-ranking, measuring how many distinct products appear in the top 5 results.

In [ ]:
# COST NOTE: 2 API calls to Claude (one expand_with_llm call per retrieval).

comparison_query = "best earbuds under 2000 rupees ANC battery life"

print(f"Comparison query: '{comparison_query}'")
print("=" * 65)

# ── Without expansion (single query, no reranking) ────────────
print("\n── WITHOUT expansion or reranking ───────────────────────────")
baseline_results = vectorstore.similarity_search_with_score(
    comparison_query, k=5
)
baseline_products = set()
for doc, score in baseline_results:
    pid = doc.metadata.get("product_id", "")
    name_line = [l for l in doc.page_content.split("\n") if l.startswith("Product:")]
    name = name_line[0].replace("Product: ", "") if name_line else "Unknown"
    if pid:
        baseline_products.add(pid)
    print(f"  Score={score:.4f} | {pid or 'policy'} | {name[:50]}")

print(f"\nDistinct products in top-5: {len(baseline_products)}")

# ── With expansion + reranking ────────────────────────────────
print("\n── WITH expansion + cross-encoder reranking ─────────────────")
# COST NOTE: API call to Claude for query expansion
expanded_candidates = multi_query_retrieve(
    comparison_query, use_llm_expansion=True, top_k_per_variant=4
)
reranked_results = rerank(comparison_query, expanded_candidates, top_n=5)

expanded_products = set()
for doc, ce_score in reranked_results:
    pid = doc.metadata.get("product_id", "")
    name_line = [l for l in doc.page_content.split("\n") if l.startswith("Product:")]
    name = name_line[0].replace("Product: ", "") if name_line else "Unknown"
    if pid:
        expanded_products.add(pid)
    print(f"  CE score={ce_score:.4f} | {pid or 'policy'} | {name[:50]}")

print(f"\nDistinct products in top-5: {len(expanded_products)}")
print(f"\nImprovement: {len(baseline_products)} → {len(expanded_products)} distinct products")
print("(More distinct products = better comparative recommendation coverage)")


In [ ]:
# ── EXPERIMENT ────────────────────────────────────────────────

# EXERCISE 1 — Hinglish query expansion
# Run expand_with_llm("sasta earphone accha battery") — a Hinglish query.
# How many variants does Claude generate? Are they useful?
# Then run multi_query_retrieve on the original vs. expanded versions.
# Does expansion improve recall for Hinglish queries?

# EXERCISE 2 — When expansion hurts
# Run multi_query_retrieve("GST registration for new seller", use_llm_expansion=True).
# Does any expanded variant retrieve product chunks instead of policy chunks?
# Add a post-filter: if the original query contains "seller" or "GST", skip expansion
# and use the original query only. Implement this heuristic and test it.

# EXERCISE 3 — Cross-encoder score calibration
# Run rerank() on the comparative query with top_n=5.
# Print the raw cross-encoder scores for all candidates.
# What is the score gap between a relevant product chunk and a policy chunk?
# Can you use this gap to build an automatic relevance threshold filter?

print("Experiments ready. Call expand_with_llm, multi_query_retrieve, rerank above.")


### Key Takeaway

Query expansion bridges the vocabulary gap between how shoppers phrase queries
("earphones") and how products are described in the catalogue ("earbuds",
"TWS"). The Claude-powered expander handles Hinglish and novel terms that the
static synonym dictionary misses. Cross-encoder re-ranking then ensures the most
relevant products appear at the top of comparative results. Together these two
techniques target the most common failure mode in CatalogueIQ: comparative
recommendation queries that retrieve only one product.

In **NB-07** you will add conversation memory using `ConversationalRetrievalChain`,
enabling the 5-turn shopper conversation where each question correctly resolves
references like "the first one you mentioned".